**1. Supervised Learning**

**Definition:** Model learns from labeled data (input → known output)

**Examples:**

Regression → predict price
Classification → spam detection

**Algorithms:**

Linear Regression
Logistic Regression
Random Forest

**Use cases:**
1. Sales prediction
2. Customer churn



**2. Unsupervised Learning**

**Definition:** No labels → model finds patterns

**Examples:**

Clustering (group users)
Dimensionality reduction

**Algorithms:**

K-Means
PCA

**Use cases:**
1. Customer segmentation
2. Anomaly detection

**3. Semi-Supervised Learning**

**Definition:** Small labeled data + large unlabeled data

Common in real-world (labeling is expensive)

**4. Reinforcement Learning**

**Definition:** Agent learns via rewards & penalties

**Used in:**

Robotics
Game AI

**Quick Comparison:**

| Type            | Data          | Goal             | Example           |
| --------------- | ------------- | ---------------- | ----------------- |
| Supervised      | Labeled       | Predict output   | Price prediction  |
| Unsupervised    | Unlabeled     | Find patterns    | Customer clusters |
| Semi-supervised | Mixed         | Improve learning | Image tagging     |
| Reinforcement   | Feedback loop | Max reward       | Game playing      |

**Data Splitting (CRITICAL CONCEPT)**
**Correct Approach**
Train → learn patterns
Validation → tune model
Test → final evaluation

**Typical split:**

70% Train
15% Validation
15% Test

**Data Leakage (Biggest Mistake)**

Definition: Model sees future/unavailable info during training

**Result:**

Unrealistically high accuracy
Model fails in real world

****DEMO: Train / Val / Test Split + Leakage Example****

In [2]:
# 1. IMPORTS

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

# 2. LOAD DATA

# Using simple dataset
from sklearn.datasets import fetch_california_housing

data = fetch_california_housing(as_frame=True)
df = data.frame

X = df.drop("MedHouseVal", axis=1)
y = df["MedHouseVal"]

# 3. CORRECT SPLIT (NO LEAKAGE)

# Step 1: Train vs Temp
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42
)

# Step 2: Validation vs Test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42
)

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

# 4. PROPER SCALING (NO LEAKAGE)

scaler = StandardScaler()

# Fit ONLY on training data
X_train_scaled = scaler.fit_transform(X_train)

# Transform others
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# 5. MODEL TRAINING

model = LinearRegression()
model.fit(X_train_scaled, y_train)

# Validation performance
val_preds = model.predict(X_val_scaled)
val_mse = mean_squared_error(y_val, val_preds)

print("Validation MSE:", val_mse)

# Final test performance
test_preds = model.predict(X_test_scaled)
test_mse = mean_squared_error(y_test, test_preds)

print("Test MSE:", test_mse)

Train: (14448, 8)
Validation: (3096, 8)
Test: (3096, 8)
Validation MSE: 0.5408750691093342
Test MSE: 0.5202604958440161


In [3]:
# BAD PRACTICE

scaler = StandardScaler()

# Fitting on FULL data before split
X_scaled = scaler.fit_transform(X)

# Split after scaling → leakage
X_train_bad, X_test_bad, y_train_bad, y_test_bad = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

model_bad = LinearRegression()
model_bad.fit(X_train_bad, y_train_bad)

preds_bad = model_bad.predict(X_test_bad)

print("Leaky MSE:", mean_squared_error(y_test_bad, preds_bad))

Leaky MSE: 0.5558915986952442


Split Strategy:
- Train: 70% → Model training
- Validation: 15% → Hyperparameter tuning
- Test: 15% → Final evaluation

Key Rule:
- No preprocessing step should use validation/test data
- All transformations fitted ONLY on training set